# <center>**Download 🚗 Images from S3** 🕸️
    
    computer-vision/datasets/cars/brands/

        source-1/
            Bmw/
                image-1
                ...
            Mercedes-Benz
            Hyundai
            Kia
            Toyota
            ...
        source-2/
            ...
        source-3/
            ...
        source-4/
            ...

---

In [1]:
import os
import boto3
from tqdm.notebook import tqdm

In [2]:
DIR = "raw-data/"

In [3]:
# S3_ACCESS_KEY_ID & S3_SECRET_ACCESS_KEY should be set as env variables
key_id = os.environ.get('S3_ACCESS_KEY_ID')
secret_key = os.environ.get('S3_SECRET_ACCESS_KEY')

if key_id is None: raise TypeError(f"'S3_ACCESS_KEY_ID' env variable is not set")
if secret_key is None: raise TypeError(f"'S3_SECRET_ACCESS_KEY' env variable is not set")

In [4]:
# enter the bucket and model name
bucket_name = input("Enter bucket name:")
model_name = input("Enter model name:")

if bucket_name == "": raise TypeError(f"'bucket_name' input variable is empty")
if model_name == "": raise TypeError(f"'model_name' input variable is empty")

Enter bucket name: computer-vision
Enter model name: brands


In [5]:
# function to get folder names in some S3 dir...
def list_folders(paginator, bucket_name, prefix):
    import boto3
    """Returns folder names in a specific path from an S3 bucket."""
    operation_parameters = {'Bucket': bucket_name, 'Prefix': prefix, 'Delimiter': '/'}

    folder_names = []
    for page in paginator.paginate(**operation_parameters):
        if 'CommonPrefixes' in page:
            for common_prefix in page['CommonPrefixes']:
                folder_name = common_prefix['Prefix'].split('/')[-2] # assumming datasets/cars/model_name/source/class
                folder_names.append(folder_name)
    return folder_names

In [6]:
%%time

# initialize session and client with boto3
session = boto3.session.Session()
s3_client = session.client('s3',
                           region_name='nyc3',
                           endpoint_url='https://nyc3.digitaloceanspaces.com',
                           aws_access_key_id=key_id,
                           aws_secret_access_key=secret_key
                          )
# initialize paginator
paginator = s3_client.get_paginator('list_objects_v2')

# for each data source download images in S3 path
sources = list_folders(paginator, bucket_name, f"datasets/cars/{model_name}/")

for source in sources:
    # get path
    path = f"datasets/cars/{model_name}/{source}/"
    print("\nConnecting to S3 in path: "+bucket_name+"/"+path)

    # get folder names 
    folder_names = list_folders(paginator, bucket_name, path)
    print(f"Found {len(folder_names)} categories.")

    folders = {}
    print("Found the following category-image pairs:")
    # for each class...
    for folder_name in folder_names:
        # ...load pages using paginator
        pages = paginator.paginate(
            Bucket=bucket_name,
            Prefix=os.path.join(path, folder_name)
        )

        # ...collect all the image keys
        folders[folder_name] = []
        for page in pages:
            for obj in page['Contents']:
                if not obj['Key'].endswith(('/')): # avoid folders
                    folders[folder_name] += [obj['Key']]
        print(f"\t - {folder_name}: {len(folders[folder_name])}")

    print(f"\nTotal of {sum(len(v) for v in folders.values())} images.")

    # download images
    for folder, img_keys in folders.items():

        # create directory
        local_dir = os.path.join(DIR, f'{source}/{folder}')
        if not os.path.exists(local_dir):
            os.makedirs(local_dir)

        # download images
        for key in tqdm(img_keys, leave=True, desc=f'Loading {folder} images'):
            local_path = os.path.join(local_dir, os.path.basename(key))
            s3_client.download_file(bucket_name, key, local_path)
    print("-"*50)


Connecting to S3 in path: computer-vision/datasets/cars/brands/source-1/
Found 5 categories.
Found the following category-image pairs:
	 - Bmw: 499
	 - Hyundai: 567
	 - Kia: 648
	 - Mercedes-Benz: 449
	 - Toyota: 568

Total of 2731 images.


Loading Bmw images:   0%|          | 0/499 [00:00<?, ?it/s]

Loading Hyundai images:   0%|          | 0/567 [00:00<?, ?it/s]

Loading Kia images:   0%|          | 0/648 [00:00<?, ?it/s]

Loading Mercedes-Benz images:   0%|          | 0/449 [00:00<?, ?it/s]

...

Loading Hyundai images:   0%|          | 0/1179 [00:00<?, ?it/s]

Loading Kia images:   0%|          | 0/692 [00:00<?, ?it/s]

Loading Mercedes-Benz images:   0%|          | 0/863 [00:00<?, ?it/s]

Loading Toyota images:   0%|          | 0/1232 [00:00<?, ?it/s]

--------------------------------------------------
CPU times: user 1min 4s, sys: 8.61 s, total: 1min 13s
Wall time: 44min 43s


---